# Naive Bayes multinomial en MapReduce sur Spark — Démonstration

Ce notebook tourne **de bout en bout sur les PETITES données (SMS Spam)**, **sans cluster**
(Spark en mode `local[*]`). Il montre :

1. le chargement et la vectorisation (bag-of-words) ;
2. l'entraînement **en RDD** (`map`/`reduceByKey`) ;
3. l'entraînement **en DataFrames** (`explode` + `groupBy`/`agg` + broadcast) ;
4. la vérification que les **deux versions donnent des prédictions identiques** ;
5. la **comparaison à `sklearn.naive_bayes.MultinomialNB`**.

> Prérequis : venv installé (`requirements.txt`) et un JDK 8/11/17. Le petit jeu SMS Spam
> est téléchargé automatiquement par la première cellule s'il est absent.

## 0. Configuration : rendre `src/` importable et télécharger les données si besoin

In [1]:
import os, sys, subprocess

# Racine du projet = dossier parent de notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
SRC = os.path.join(ROOT, "src")
sys.path.insert(0, SRC)

# Télécharger le petit jeu SMS Spam s'il n'existe pas encore.
sms_csv = os.path.join(ROOT, "data", "sms_spam.csv")
if not os.path.exists(sms_csv):
    print("SMS Spam absent -> téléchargement ...")
    subprocess.run([sys.executable, os.path.join(ROOT, "data", "download_sms_spam.py")], check=True)
print("Données :", sms_csv, "->", "OK" if os.path.exists(sms_csv) else "MANQUANT")

Données : /Users/abdrahamanecamara/Documents/Dauphine/Projet ML/data/sms_spam.csv -> OK


## 1. Chargement et préparation des données

On charge les SMS (étiquettes `ham`/`spam`), on découpe en train/test (stratifié,
reproductible), puis on **tokenise** et on construit le **vocabulaire** sur l'entraînement.
Chaque document devient une liste d'indices de mots (bag-of-words).

In [2]:
import nb_common as C

texts, labels = C.load_sms_spam(sms_csv)
print(f"{len(texts)} messages | spam={labels.count('spam')} ham={labels.count('ham')}")

# Découpage train/test reproductible (80/20, stratifié).
X_tr, X_te, y_tr, y_te = C.train_test_split_texts(texts, labels, test_size=0.2, seed=42)

# Tokenisation + vocabulaire + vectorisation en indices de mots (partagé par les 2 versions).
data = C.prepare(X_tr, y_tr, X_te, y_te)
print(f"Vocabulaire : {data.vocab_size} mots")
print(f"Train / Test : {len(data.train)} / {len(data.test)} documents")
print("Classes :", data.idx_to_label)

# Exemple : un document vectorisé (classe, premiers indices de mots)
c0, w0 = data.train[0]
print("Exemple doc -> classe", data.idx_to_label[c0], "| indices mots:", w0[:15], "...")

5574 messages | spam=747 ham=4827


Vocabulaire : 7716 mots
Train / Test : 4459 / 1115 documents
Classes : ['ham', 'spam']
Exemple doc -> classe ham | indices mots: [4917, 947, 4936, 6783, 7376, 6900, 3496, 3446, 3446] ...


## 2. Démarrage de Spark (local, sans cluster)

`get_spark` sélectionne automatiquement un JDK compatible (17) et expédie nos modules
Python aux workers. Le mode `local[*]` utilise tous les cœurs de la machine.

In [3]:
spark = C.get_spark(app_name="demo-sms", master="local[*]")
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
print("Spark", spark.version, "| parallélisme =", sc.defaultParallelism, "cœurs")

26/07/08 21:23:35 WARN Utils: Your hostname, Macbook-Abdrahamane.local resolves to a loopback address: 127.0.0.1; using 192.168.0.32 instead (on interface en0)
26/07/08 21:23:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/08 21:23:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.8 | parallélisme = 8 cœurs


## 3. Version RDD — `map` / `reduceByKey`

Le cœur MapReduce : pour chaque occurrence d'un mot `w` dans un document de classe `c`,
on émet `((c, w), 1)` (**map**), puis `reduceByKey(add)` additionne les occurrences
(**reduce**) pour obtenir `count_{w,c}`. Le modèle (log-probas) est ensuite construit
puis **broadcasté** pour la prédiction.

In [4]:
import nb_rdd

model_rdd = nb_rdd.train_rdd(sc, data.train, data.vocab_size, data.idx_to_label)
acc_rdd = nb_rdd.evaluate_rdd(sc, model_rdd, data.test)
print(f"[RDD] accuracy test = {acc_rdd:.4f}")

# Aperçu du modèle : priors log P(c)
for c, lab in enumerate(model_rdd.idx_to_label):
    print(f"  log P({lab}) = {model_rdd.log_prior[c]:.4f}")

[RDD] accuracy test = 0.9848
  log P(ham) = -0.1440
  log P(spam) = -2.0091


## 4. Version DataFrames — `explode` / `groupBy`-`agg`

Même algorithme, exprimé en relationnel : `explode(words)` produit une ligne par mot
(**map**), `groupBy(label, word).count()` agrège (**reduce**). La prédiction utilise une
**UDF** s'appuyant sur le modèle **broadcasté**.

In [5]:
import nb_dataframe

model_df = nb_dataframe.train_dataframe(spark, data.train, data.vocab_size, data.idx_to_label)
acc_df = nb_dataframe.evaluate_dataframe(spark, model_df, data.test)
print(f"[DataFrame] accuracy test = {acc_df:.4f}")

[DataFrame] accuracy test = 0.9848


## 5. Vérification : RDD et DataFrames donnent des prédictions identiques

In [6]:
pred_rdd = nb_rdd.predict_rdd(sc, model_rdd, data.test)
pred_df = nb_dataframe.predict_dataframe(spark, model_df, data.test)

print("Accuracy identique :", acc_rdd == acc_df, f"({acc_rdd:.4f})")
print("Prédictions identiques (ligne à ligne) :", pred_rdd == pred_df)
assert pred_rdd == pred_df, "Les deux versions doivent prédire exactement pareil !"
print("OK ✅")

Accuracy identique : True (0.9848)
Prédictions identiques (ligne à ligne) : True
OK ✅


## 6. Comparaison avec `sklearn.naive_bayes.MultinomialNB`

On entraîne sklearn avec **le même vocabulaire et la même tokenisation**, et le même
lissage (`alpha=1`). Notre implémentation reproduit MultinomialNB : les prédictions
doivent coïncider.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Même vocabulaire (celui construit sur le train) + même tokenizer que nos versions Spark.
cv = CountVectorizer(tokenizer=C.tokenize, vocabulary=data.vocab,
                     token_pattern=None, lowercase=False)
Xtr = cv.transform(X_tr)
Xte = cv.transform(X_te)

y_tr_idx = [data.label_to_idx[l] for l in y_tr]
y_te_idx = [data.label_to_idx[l] for l in y_te]

clf = MultinomialNB(alpha=1.0)
clf.fit(Xtr, y_tr_idx)
pred_sk = list(clf.predict(Xte))

acc_sk = accuracy_score(y_te_idx, pred_sk)
print(f"[sklearn]   accuracy test = {acc_sk:.4f}")
print(f"[RDD/DF]    accuracy test = {acc_rdd:.4f}")
print("Prédictions Spark == sklearn :", [int(p) for p in pred_rdd] == [int(p) for p in pred_sk])

[sklearn]   accuracy test = 0.9848
[RDD/DF]    accuracy test = 0.9848
Prédictions Spark == sklearn : True


## 7. Prédiction sur de nouveaux messages

On applique le modèle RDD (identique au DataFrame) à quelques messages inédits.

In [8]:
nouveaux = [
    "Congratulations! You have WON a free prize, claim now!!!",
    "Hey, are we still on for lunch tomorrow?",
    "URGENT: your account needs verification, click this link to win cash",
    "Can you send me the slides before the meeting?",
]
# Vectorisation identique au pipeline d'entraînement.
new_indices = [C.doc_to_indices(C.tokenize(t), data.vocab) for t in nouveaux]
for txt, idxs in zip(nouveaux, new_indices):
    c = C.predict_indices(model_rdd, idxs)
    print(f"[{data.idx_to_label[c]:>4}]  {txt}")

[spam]  Congratulations! You have WON a free prize, claim now!!!
[ ham]  Hey, are we still on for lunch tomorrow?
[spam]  URGENT: your account needs verification, click this link to win cash
[ ham]  Can you send me the slides before the meeting?


In [9]:
spark.stop()
print("SparkSession arrêtée.")

SparkSession arrêtée.
